# Class Exercise: Structured Output with Gemini

<small>OPAN 6604 - Week 4: this code uses the Gemini API. Paste your key when prompted - it stays in memory only.</small>

<small>**Task:** Practice the two structured-output modes from the demo: force the model to return exactly one label, then a single typed object.

- **A)** Classify movie reviews into a 1 to 5 star rating using enum mode.
- **B)** Given a few movies and a viewer request, recommend the single best movie as a typed pick (title and reason).
- **C)** Short answer: when to use enum vs JSON schema vs plain text, and why structured output beats parsing free text.</small>

### Step 0: Setup, connect to Gemini

In [2]:
import getpass
from enum import Enum
from google import genai
from pydantic import BaseModel, Field

# Paste your key when prompted. It stays in memory only and is never saved in the notebook.
client = genai.Client(api_key=getpass.getpass("Enter your Gemini API key: "))
MODEL = "gemini-2.5-flash-lite"

### A) Classify reviews into a 1 to 5 star rating

<small>Define a `StarRating` enum with five members (`ONE` to `FIVE`, values `"1"` to `"5"`). For each review, call `generate_content` with `response_mime_type="text/x.enum"` and `response_schema=StarRating`, then print the rating next to the review.

Hint: enum members can be strings, e.g. `ONE = "1"`. This is the same enum mode as the demo's `Sentiment` classifier.</small>

In [3]:
reviews = [
    "An absolute masterpiece, I was on the edge of my seat the whole time.",
    "It was fine. A few good moments but mostly forgettable.",
    "I walked out halfway through. A complete waste of time.",
]

# A 1 to 5 rating is a small fixed set, so an enum is the natural schema.
class StarRating(Enum):
    ONE = "1"
    TWO = "2"
    THREE = "3"
    FOUR = "4"
    FIVE = "5"

for r in reviews:
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Rate this movie review from 1 to 5 stars: {r}",
        config={"response_mime_type": "text/x.enum",
                "response_schema": StarRating},
    )
    print(f" {r} -> {response.text} stars")

 An absolute masterpiece, I was on the edge of my seat the whole time. -> 5 stars
 It was fine. A few good moments but mostly forgettable. -> 3 stars
 I walked out halfway through. A complete waste of time. -> 5 stars


### B) Recommend a movie for a viewer request

<small>Define a Pydantic model `Pick` with `title` and `reason` fields. Given the `movies` list and the `viewer_request`, call `generate_content` with `response_mime_type="application/json"` and `response_schema=Pick`, then print the single best pick from `response.parsed`.

Hint: build a text catalog of the movies for the prompt and put the task in a `system_instruction`, the same shape as the demo's recommender.</small>

In [4]:
movies = [
    {"title": "The Grand Budapest Hotel (2014)", "genres": "Comedy, Drama"},
    {"title": "Sicario (2015)", "genres": "Action, Crime, Thriller"},
    {"title": "Paddington 2 (2017)", "genres": "Comedy, Family"},
    {"title": "Hereditary (2018)", "genres": "Drama, Horror, Mystery"},
    {"title": "La La Land (2016)", "genres": "Comedy, Drama, Romance, Musical"},
]
viewer_request = "something light and cheerful for a family movie night"

# Pick is the output schema: the model fills these fields in for the movie it returns.
class Pick(BaseModel):
    title: str = Field(description="Exact movie title from the list.")
    reason: str = Field(description="One sentence on why it fits the viewer's request.")

catalog = "\n".join(f"- {m['title']} [{m['genres']}]" for m in movies)

# response_schema=Pick returns one object: the single best movie. Alternatively, you could
# ask for response_schema=list[Pick] to rank them all and then show only response.parsed[0].
response = client.models.generate_content(
    model=MODEL,
    contents=f"Viewer's request: {viewer_request}\n\nMovies:\n{catalog}",
    config={
        "system_instruction": (
            "You are a movie concierge. From the list, recommend the single best movie for the "
            "viewer's request, with a one-sentence reason. Use the exact title from the list."
        ),
        "response_mime_type": "application/json",
        "response_schema": Pick,
    },
)

pick = response.parsed
print(f"{pick.title}: {pick.reason}")

Paddington 2: This is a delightful and heartwarming film that is perfect for all ages and guaranteed to bring smiles.


### C) When would you use each output mode?

<small>In a few sentences: when is enum mode the right choice, when do you need a JSON schema (a list of typed objects), and when is plain text fine? Why is forcing a structure more reliable than asking for an answer in plain text and parsing it yourself?</small>

<small>**Answer (sample).** Enum mode fits when the output must be exactly one label from a small fixed set (sentiment, a star rating, a yes or no), so the result drops straight into code with no parsing. A JSON schema (a list of typed objects) fits when you need several fields per item or a whole list, like a ranked set of picks each with a reason, which is the recommender pattern. Plain text is fine when the output is meant for a person to read and is not consumed by code. Forcing a structure is more reliable because the model cannot drift off format: with an enum it can only return one of the allowed values, and with a schema every field is present and typed, so you avoid brittle string parsing and the errors that appear when the model phrases its answer differently each run.</small>